# Contrib Commands


## Qué son?
    
    Son wrappers fáciles de usar de comandos de shell complejos.

In [9]:
import sh
from sh.contrib import sudo
from sh import Command
ls = Command("ls")
my_password = "Agustin18898\n"

## Usando sudo con contrib vs sin contrib

In [6]:
with sudo:
    print(ls("/"))

[sudo] password for aborda:  ········


.VolumeIcon.icns	System			home
.file			Users			opt
.nofollow		Volumes			private
.resolve		bin			sbin
.vol			cores			tmp
Applications		dev			usr
Library			etc			var



In [10]:
my_sudo = sh.sudo.bake("-S", _in=my_password)
print(my_sudo.ls("/"))

.VolumeIcon.icns	System			home
.file			Users			opt
.nofollow		Volumes			private
.resolve		bin			sbin
.vol			cores			tmp
Applications		dev			usr
Library			etc			var



## Pero y si no quiero poner mi contraseña cada vez?

In [12]:
with sudo(password=my_password, _with=True):
    print(ls("/"))

.VolumeIcon.icns	System			home
.file			Users			opt
.nofollow		Volumes			private
.resolve		bin			sbin
.vol			cores			tmp
Applications		dev			usr
Library			etc			var



## Me puedo definir mis propios wrappers?
    Se pueden agregar comandos contrib utilizando el decorator "@contrib", por ejemplo:

In [33]:
from sh import contrib
@contrib("echo")
def echo(original):
    return original.bake("-n")


In [34]:
from sh.contrib import echo
echo("Hello!")

'Hello!'

## El patrón decorator

![title](resources/decorator.png)

## El patrón decorator

Mientras tengamos en mente este patrón de diseño, es fácil agregar contrib commands. Sin embargo este módulo tiene algunas limitaciones:

- Solamente es posible crear contrib commands de comandos ya existentes.
- Por cómo funciona el módulo, solo se pueden decorar los comandos una sola vez.

## Cómo funciona este módulo?
    Código del módulo Contrib:

In [ ]:
#Nota: no me ejecutes! Soy código robado!
class Contrib(ModuleType):  # pragma: no cover
    @classmethod
    def __call__(cls, name):
        def wrapper1(fn):
            @property
            def cmd_getter(self):
                cmd = resolve_command(name, Command)

                if not cmd:
                    raise CommandNotFound(name)

                new_cmd = fn(cmd)
                return new_cmd

            setattr(cls, name, cmd_getter)
            return fn

        return wrapper1


mod_name = __name__ + ".contrib"
contrib = Contrib(mod_name)
sys.modules[mod_name] = contrib


## Muy bonito, pero que hace?

Vamos por partes

In [ ]:
            def cmd_getter(self):
                cmd = resolve_command(name, Command)

                if not cmd:
                    raise CommandNotFound(name)

                new_cmd = fn(cmd)
                return new_cmd

Esto nos busca el comando original en nuestro sistema y lo decora con la función que le pasamos, ejecutándose primero la nueva función y después el comando original.

## Muy bonito, pero que hace?

In [ ]:
        def wrapper1(fn):
            @property
            ...

            setattr(cls, name, cmd_getter)
            return fn

Luego esta parte crea un nuevo atributo con el nombre del comando en el objeto contrib, cuyo valor es el Command decorado!

In [37]:
getattr(contrib,"echo")

<Command '/bin/echo -n'>

## Muy bonito, pero que hace?

In [ ]:
class Contrib(ModuleType):  # pragma: no cover
    @classmethod
    def __call__(cls, name):
        ...

        return wrapper1

Esto nos convierte la clase en ejecutable, lo que nos permite utilizar la sintaxis de operadores en python  

In [ ]:
mod_name = __name__ + ".contrib"
contrib = Contrib(mod_name)
sys.modules[mod_name] = contrib

Finalmente, creamos un objeto de la clase Contrib y lo agregamos a los módulos del sistema.